# Final revision — Step 5

Run this notebook from this copied project folder, or set `NETWORK_ANALYSIS_LOCAL_ROOT` to its absolute path. Run Steps 01–07 in separate kernels. Existing outputs in this copy may be overwritten; the source repository is untouched.

The approved hypotheses, thresholds, seeds, null ensembles and removal protocol are retained. Cleared outputs avoid presenting historical execution as validation of revised code. Step 04 is the full historical generation workflow and is computationally expensive; the supplied cached ensembles allow Steps 05–07 to run without regenerating it. Consult `VALIDATION.md` for exactly what was rerun.


# Step 5: Airport-removal experiment setup

**Scope.** This notebook prepares the approved removal experiments on the observed networks and the saved randomized networks. It calculates removal counts, targeted rankings, paired random orders and tie information. **It does not calculate any resilience metric, compare observed with null networks, assign H1 or H2 verdicts, or run Step 7.**

| Decision | Rule |
|---|---|
| Removal levels | 5, 10, 15 and 20 percentage points |
| Removal count | K = `(2 * N * p + 100) // 200`, with p in percentage points |
| Degree targeting | Descending total degree; exact ties by ascending BTS airport ID |
| Betweenness targeting | Descending exact directed hop-based betweenness; numerical ties (equal after rounding to 12 decimals) by ascending BTS airport ID |
| Rankings | Calculated on each intact network, fixed throughout its removal sequence, recomputed for each randomized network |
| Random removal | The saved 100 airport orders per airline, paired by trial number across the observed and all randomized networks |
| Tie-order sensitivity | 100 random orders within tied groups on observed networks, for betweenness and degree; ID-ranking results remain official |

## Section 5.0: Setup

In [1]:
import hashlib
import os
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd

# Local execution is enabled only when this environment variable is set.
# It is not set on Colab, so Colab always uses the Google Drive path below.
LOCAL_PROJECT_ROOT = os.environ.get("NETWORK_ANALYSIS_LOCAL_ROOT")

if LOCAL_PROJECT_ROOT:
    BASE = Path(LOCAL_PROJECT_ROOT).expanduser().resolve() / "Data"
elif (Path.cwd() / "Data").is_dir():
    BASE = Path.cwd() / "Data"
else:
    from google.colab import drive
    drive.mount("/content/drive")

    # Notice the trailing space after 'Network Analysis '
    BASE = Path("/content/drive/MyDrive/Network Analysis /Data")

PROCESSED = BASE / "Processed"
NULLS = PROCESSED / "Nulls"
PLANS = PROCESSED / "Removal_Plans"
PROJECT_ROOT = BASE.parent
TABLES = PROJECT_ROOT / "Results" / "Tables"
PLANS.mkdir(parents=True, exist_ok=True)

AIRLINES = ["AA", "DL", "WN"]
AIRLINE_INDEX = {a: i for i, a in enumerate(AIRLINES)}
MASTER_SEED = 20250401
LEVELS = [5, 10, 15, 20]
STRATEGY_INDEX = {"Degree": 0, "Betweenness": 1}
STAGE_RANDOM_ORDERS = 5
STAGE_TIE_ORDERS = 7
EXPECTED_PRIMARY = 200
EXPECTED_SUPPLEMENTARY = 100
EXPECTED_RANDOM_ORDERS = 100
TIE_ORDER_REPLICATES = 100
CHECK_APPROVED_TIES = True

checks = []
seed_rows = []


class StopProtocolError(RuntimeError):
    pass


def check(name, condition, detail=""):
    if not bool(condition):
        pd.DataFrame(checks).to_csv(TABLES / "05_setup_validation.csv", index=False)
        raise StopProtocolError(f"FAILED: {name}. {detail}")
    checks.append({"Check": name, "Result": "PASS", "Detail": str(detail)})


def removal_count(n, p):
    """p is in percentage points (5, 10, 15 or 20); halves round up."""
    return (2 * n * p + 100) // 200


def network_sha(source, target, n):
    return hashlib.sha256(np.sort(source * n + target).astype("<i8").tobytes()).hexdigest()


OBS = {}

for airline in AIRLINES:
    nodes = pd.read_csv(PROCESSED / f"{airline}_nodes.csv")
    edges = pd.read_csv(PROCESSED / f"{airline}_edges.csv")
    ids = np.sort(nodes["airport_id"].to_numpy(dtype=np.int64))
    n = len(ids)
    source = np.searchsorted(ids, edges["source"].to_numpy(dtype=np.int64))
    target = np.searchsorted(ids, edges["target"].to_numpy(dtype=np.int64))
    OBS[airline] = {
        "n": n, "m": len(source), "ids": ids, "source": source, "target": target,
        "codes": nodes.set_index("airport_id").loc[ids, "code"].to_numpy(),
        "in": np.bincount(target, minlength=n), "out": np.bincount(source, minlength=n)
    }

expected_K = {"AA": [12, 23, 35, 46], "DL": [11, 21, 32, 42], "WN": [5, 10, 16, 21]}
LEVEL_K = {a: {p: removal_count(OBS[a]["n"], p) for p in LEVELS} for a in AIRLINES}
count_rows = []

for airline in AIRLINES:
    check(f"{airline}: removal counts match the approved table", [LEVEL_K[airline][p] for p in LEVELS] == expected_K[airline], LEVEL_K[airline])
    for p in LEVELS:
        count_rows.append({"Airline": airline, "Airports N": OBS[airline]["n"], "Level (percentage points)": p,
                           "Removed airports K": LEVEL_K[airline][p], "Realized fraction (%)": 100 * LEVEL_K[airline][p] / OBS[airline]["n"]})

removal_counts_df = pd.DataFrame(count_rows)
removal_counts_df.to_csv(TABLES / "05_removal_counts.csv", index=False)
display(removal_counts_df.round(2))

,Airline,Airports N,Level (percentage points),Removed airports K,Realized fraction (%)
0,AA,230,5,12,5.22
1,AA,230,10,23,10.00
2,AA,230,15,35,15.22
3,AA,230,20,46,20.00
4,DL,212,5,11,5.19
5,DL,212,10,21,9.91
6,DL,212,15,32,15.09
7,DL,212,20,42,19.81
8,WN,104,5,5,4.81
9,WN,104,10,10,9.62


## Section 5.1: Networks and manifest

Saved networks are read, not regenerated. Each network's content hash is compared with the hash recorded when it was generated.

In [2]:
NETWORKS = {}
manifest_rows = []
prior_records = pd.read_csv(TABLES / "04_production_chain_records.csv")
expansion_records = pd.read_csv(TABLES / "04_production_expansion_chain_records.csv")
recorded_hash = {
    (r["Airline"], r["Model"], int(r["Network"])): r["Network SHA-256"]
    for r in pd.concat([prior_records, expansion_records], ignore_index=True).to_dict("records")
}


airport_membership_checked = 0


def to_indices(ids, source_ids, target_ids):
    global airport_membership_checked
    assert np.isin(source_ids, ids).all() and np.isin(target_ids, ids).all()
    airport_membership_checked += source_ids.shape[0]
    return np.searchsorted(ids, source_ids), np.searchsorted(ids, target_ids)


def add_network(airline, model, network, source, target, filename):
    obs = OBS[airline]
    sha = network_sha(source, target, obs["n"])
    expected = recorded_hash.get((airline, model, network))
    matches = (expected == sha) if expected is not None else True
    manifest_rows.append({"Airline": airline, "Network set": model, "Network": network, "Source file": filename,
                          "Network SHA-256": sha, "Matches recorded hash": matches})
    if not matches:
        raise StopProtocolError(f"{airline} {model} network {network}: hash does not match its recorded value.")
    NETWORKS[(airline, model, network)] = (source, target)


for airline in AIRLINES:
    obs = OBS[airline]
    add_network(airline, "observed", 0, obs["source"], obs["target"], f"{airline}_edges.csv")

    stored = np.load(NULLS / f"{airline}_directed_null_networks.npz")
    source, target = to_indices(obs["ids"], stored["source"], stored["target"])
    for i, network in enumerate(range(source.shape[0])):
        add_network(airline, "directed", network, source[i], target[i], f"{airline}_directed_null_networks.npz")

    extra = np.load(NULLS / f"{airline}_directed_null_networks_expansion_chains_100_199.npz")
    source, target = to_indices(obs["ids"], extra["source"], extra["target"])
    for i, network in enumerate(extra["chain_ids"].tolist()):
        add_network(airline, "directed", int(network), source[i], target[i], f"{airline}_directed_null_networks_expansion_chains_100_199.npz")

    stored = np.load(NULLS / f"{airline}_reciprocal_null_networks.npz")
    source, target = to_indices(obs["ids"], stored["source"], stored["target"])
    for i, network in enumerate(range(source.shape[0])):
        add_network(airline, "reciprocal", network, source[i], target[i], f"{airline}_reciprocal_null_networks.npz")

network_manifest_df = pd.DataFrame(manifest_rows)
network_manifest_df.to_csv(TABLES / "05_network_manifest.csv", index=False)

counts = network_manifest_df.groupby(["Airline", "Network set"]).size().unstack()
display(counts)
check("Observed networks present", (counts["observed"] == 1).all(), counts["observed"].to_dict())
check("Primary-null network counts", (counts["directed"] == EXPECTED_PRIMARY).all(), counts["directed"].to_dict())
check("Supplementary-null network counts", (counts["reciprocal"] == EXPECTED_SUPPLEMENTARY).all(), counts["reciprocal"].to_dict())
check("Every randomized network matches its recorded hash", network_manifest_df["Matches recorded hash"].all(), len(network_manifest_df))

for (airline, model, network), (source, target) in NETWORKS.items():
    obs = OBS[airline]
    assert np.array_equal(np.bincount(source, minlength=obs["n"]), obs["out"]) and np.array_equal(np.bincount(target, minlength=obs["n"]), obs["in"])
check("Every network keeps the observed in-degree and out-degree of each airport", True, len(NETWORKS))

Network set,directed,observed,reciprocal
Airline,,,
AA,200,1,100
DL,200,1,100
WN,200,1,100


## Section 5.2: Targeted rankings

Rankings are calculated once on each intact network and stored. They are not updated during removal. Observed betweenness reuses the accepted Step 2 values, which are also checked against a fresh calculation.

Degree ties are exact. Betweenness values equal after rounding to 12 decimals are **numerical ties**.

In [3]:
def betweenness_of(source, target, n):
    G = nx.DiGraph()
    G.add_nodes_from(range(n))
    G.add_edges_from(zip(source.tolist(), target.tolist()))
    scores = nx.betweenness_centrality(G, normalized=True, weight=None)
    return np.array([scores[i] for i in range(n)], dtype=float)


def targeted_order(values, ids):
    """Descending value, with ties broken by ascending BTS airport ID."""
    return np.lexsort((ids, -values))


def cutoff_ties(values, order, k):
    cut = values[order[k - 1]]
    group = int((values == cut).sum())
    above = int((values > cut).sum())
    selected = k - above
    return {"Tied group at cut-off": group, "Selected from tie": selected, "Left in tie": group - selected,
            "Tie straddles cut-off": group - selected > 0 and selected > 0,
            "Cut-off value is zero": bool(cut == 0)}


step2 = pd.read_csv(TABLES / "02_airport_centralities.csv")
RANKINGS = {}
tie_rows, verification_rows = [], []

for (airline, model, network), (source, target) in NETWORKS.items():
    obs = OBS[airline]
    n, ids = obs["n"], obs["ids"]
    degree_values = (obs["in"] + obs["out"]).astype(float)

    if model == "observed":
        accepted = step2[step2["Airline"] == airline].sort_values("AirportID")
        assert np.array_equal(accepted["AirportID"].to_numpy(), ids)
        raw_betweenness = accepted["Betweenness"].to_numpy(dtype=float)
        fresh = betweenness_of(source, target, n)
        verification_rows.append({"Airline": airline, "Maximum absolute difference (Step 2 vs fresh)": float(np.abs(raw_betweenness - fresh).max()),
                                  "Same ranking": bool(np.array_equal(targeted_order(np.round(raw_betweenness, 12), ids), targeted_order(np.round(fresh, 12), ids)))})
    else:
        raw_betweenness = betweenness_of(source, target, n)

    rounded = np.round(raw_betweenness, 12)
    degree_order = targeted_order(degree_values, ids)
    betweenness_order = targeted_order(rounded, ids)
    RANKINGS[(airline, model, network)] = {"Degree": degree_order, "Betweenness": betweenness_order,
                                          "Degree values": degree_values, "Betweenness values": rounded, "Raw betweenness": raw_betweenness}

    for strategy, values, order in [("Degree", degree_values, degree_order), ("Betweenness", rounded, betweenness_order)]:
        for p, k in LEVEL_K[airline].items():
            tie_rows.append({"Airline": airline, "Network set": model, "Network": network, "Strategy": strategy,
                             "Level": p, "K": k, **cutoff_ties(values, order, k)})

ties_df = pd.DataFrame(tie_rows)
ties_df.to_csv(TABLES / "05_cutoff_ties_by_network.csv", index=False)
verification_df = pd.DataFrame(verification_rows)
display(verification_df)
check("Step 2 observed betweenness matches a fresh calculation", (verification_df["Maximum absolute difference (Step 2 vs fresh)"] <= 1e-12).all() and verification_df["Same ranking"].all())

for airline in AIRLINES:
    obs = OBS[airline]
    subset = {key: value for key, value in NETWORKS.items() if key[0] == airline}

    for model in ["observed", "directed", "reciprocal"]:
        keys = sorted([key for key in subset if key[1] == model], key=lambda item: item[2])
        degree = np.vstack([obs["ids"][RANKINGS[key]["Degree"]] for key in keys])
        between = np.vstack([obs["ids"][RANKINGS[key]["Betweenness"]] for key in keys])
        values = np.vstack([RANKINGS[key]["Raw betweenness"] for key in keys])
        np.savez_compressed(
            PLANS / f"{airline}_{model}_targeted_rankings.npz",
            airport_ids=obs["ids"], network_ids=np.array([key[2] for key in keys]),
            degree_order=degree, betweenness_order=between, betweenness_values=values
        )

for key, ranking in RANKINGS.items():
    airline = key[0]
    ids = OBS[airline]["ids"]
    for strategy in ("Degree", "Betweenness"):
        order = ranking[strategy]
        values = ranking["Degree values"] if strategy == "Degree" else ranking["Betweenness values"]
        assert len(order) == len(ids) and len(set(order.tolist())) == len(ids)
        ordered_values = values[order]
        assert (ordered_values[:-1] >= ordered_values[1:]).all()
        tie_positions = np.flatnonzero(ordered_values[:-1] == ordered_values[1:])
        assert (ids[order[tie_positions]] < ids[order[tie_positions + 1]]).all()

check("Every ranking is a full airport order, descending, with ties by ascending BTS ID", True, f"{2 * len(RANKINGS)} rankings")

degree_same = all(
    np.array_equal(RANKINGS[key]["Degree"], RANKINGS[(key[0], "observed", 0)]["Degree"]) for key in RANKINGS
)
check("Degree rankings are identical across each airline's networks, as degrees are preserved", degree_same)

,Airline,Maximum absolute difference (Step 2 vs fresh),Same ranking
0,AA,9.367507e-17,True
1,DL,9.020562e-17,True
2,WN,9.714451e-17,True


In [4]:
# Approved observed cut-off ties: (selected from tie, tied group). All other cases do not straddle a cut-off.
approved_ties = {
    ("AA", "Degree", 10): (2, 5), ("AA", "Degree", 15): (9, 16), ("AA", "Degree", 20): (4, 14),
    ("AA", "Betweenness", 10): (1, 3), ("AA", "Betweenness", 15): (4, 7), ("AA", "Betweenness", 20): (8, 192),
    ("DL", "Degree", 15): (1, 15), ("DL", "Degree", 20): (11, 15),
    ("DL", "Betweenness", 10): (1, 5), ("DL", "Betweenness", 20): (10, 19),
}
observed_ties = ties_df[ties_df["Network set"] == "observed"]

if CHECK_APPROVED_TIES:
    for row in observed_ties.itertuples(index=False):
        key = (row.Airline, row.Strategy, row.Level)
        if key in approved_ties:
            assert row[9] is True or row[9] == True, key
            assert (row[7], row[6]) == approved_ties[key], (key, row[7], row[6])
        else:
            assert row[9] == False, key
    check("Observed cut-off ties match the approved protocol table", True, len(approved_ties))

display(observed_ties[observed_ties["Tie straddles cut-off"]].drop(columns=["Network set", "Network"]).reset_index(drop=True))

# The selected zero-betweenness airports under ascending ID for AA at 20%.
aa = OBS["AA"]
aa_order = RANKINGS[("AA", "observed", 0)]["Betweenness"]
aa_values = RANKINGS[("AA", "observed", 0)]["Betweenness values"]
zero_selected = [i for i in aa_order[:LEVEL_K["AA"][20]] if aa_values[i] == 0]
aa_zero = pd.DataFrame({
    "Rank": [int(np.where(aa_order == i)[0][0]) + 1 for i in zero_selected],
    "BTS airport ID": aa["ids"][zero_selected], "Code": aa["codes"][zero_selected],
    "Degree": (aa["in"][zero_selected] + aa["out"][zero_selected]) / 2
})
aa_zero.to_csv(TABLES / "05_aa_zero_betweenness_selected_at_20.csv", index=False)
display(aa_zero)
check("AA at 20% selects eight zero-betweenness airports by ascending ID", len(aa_zero) == 8, list(aa_zero["Code"]))

step4_ties = pd.concat([
    pd.read_csv(TABLES / "04_null_betweenness_ties_at_cutoffs.csv"),
    pd.read_csv(TABLES / "04_null_betweenness_ties_at_cutoffs_chains_100_199.csv")
], ignore_index=True).rename(columns={"Cut-off betweenness is zero": "Step 4 zero"})
current = ties_df[(ties_df["Network set"] == "directed") & (ties_df["Strategy"] == "Betweenness")]
comparison = current.merge(step4_ties, on=["Airline", "Network", "Level"], how="outer", suffixes=("", " Step 4"), validate="one_to_one", indicator=True)
check("Betweenness ties match the saved Step 4 records for every primary-null network", (comparison["_merge"] == "both").all()
      and (comparison["Tied group at cut-off"] == comparison["Tied group at cut-off Step 4"]).all()
      and (comparison["Selected from tie"] == comparison["Selected from tie Step 4"]).all()
      and (comparison["Cut-off value is zero"] == comparison["Step 4 zero"]).all(), len(comparison))

summary = ties_df[ties_df["Network set"].isin(["directed", "reciprocal"])].groupby(["Airline", "Network set", "Strategy", "Level"]).agg(
    Networks=("Network", "count"), K=("K", "first"),
    Mean_tied_group=("Tied group at cut-off", "mean"), Maximum_tied_group=("Tied group at cut-off", "max"),
    Mean_selected_from_tie=("Selected from tie", "mean"), Share_straddling=("Tie straddles cut-off", "mean"),
    Networks_with_zero_cutoff=("Cut-off value is zero", "sum")
).reset_index()
summary.to_csv(TABLES / "05_cutoff_tie_summary.csv", index=False)
display(summary[summary["Strategy"] == "Betweenness"].round(2))

,Airline,Strategy,Level,K,Tied group at cut-off,Selected from tie,Left in tie,Tie straddles cut-off,Cut-off value is zero
0,AA,Degree,10,23,5,2,3,True,False
1,AA,Degree,15,35,16,9,7,True,False
2,AA,Degree,20,46,14,4,10,True,False
3,AA,Betweenness,10,23,3,1,2,True,False
4,AA,Betweenness,15,35,7,4,3,True,False
5,AA,Betweenness,20,46,192,8,184,True,True
6,DL,Degree,15,32,15,1,14,True,False
7,DL,Degree,20,42,15,11,4,True,False
8,DL,Betweenness,10,21,5,1,4,True,False
9,DL,Betweenness,20,42,19,10,9,True,False


,Rank,BTS airport ID,Code,Degree
0,39,10135,ABE,1.0
1,40,10136,ABI,1.0
2,41,10140,ABQ,4.0
3,42,10154,ACK,5.0
4,43,10155,ACT,1.0
5,44,10185,AEX,1.0
6,45,10257,ALB,5.0
7,46,10268,ALO,1.0


,Airline,Network set,Strategy,Level,Networks,K,Mean_tied_group,Maximum_tied_group,Mean_selected_from_tie,Share_straddling,Networks_with_zero_cutoff
0,AA,directed,Betweenness,5,200,12,1.00,2,1.00,0.00,0
1,AA,directed,Betweenness,10,200,23,1.15,7,1.06,0.08,0
2,AA,directed,Betweenness,15,200,35,190.34,206,6.00,0.96,189
3,AA,directed,Betweenness,20,200,46,198.94,206,16.73,1.00,198
8,AA,reciprocal,Betweenness,5,100,12,1.02,3,1.02,0.00,0
9,AA,reciprocal,Betweenness,10,100,23,96.89,211,1.51,0.58,46
10,AA,reciprocal,Betweenness,15,100,35,207.29,211,12.29,1.00,100
11,AA,reciprocal,Betweenness,20,100,46,207.29,211,23.29,1.00,100
16,DL,directed,Betweenness,5,200,11,1.01,2,1.00,0.02,0
17,DL,directed,Betweenness,10,200,21,1.00,2,1.00,0.00,0


## Section 5.3: Paired random removal orders

The accepted 100 airport orders per airline are reused. The file holds one set of orders per airline, so trial t means the same order on the observed network and every randomized network. Each order is regenerated from its recorded seed and compared with the saved file.

In [5]:
orders_df = pd.read_csv(TABLES / "04_random_removal_orders.csv").sort_values(["Airline", "Trial"]).reset_index(drop=True)
random_rows = []

for airline in AIRLINES:
    obs = OBS[airline]
    group = orders_df[orders_df["Airline"] == airline]
    check(f"{airline}: 100 random orders saved", len(group) == EXPECTED_RANDOM_ORDERS and sorted(group["Trial"]) == list(range(EXPECTED_RANDOM_ORDERS)), len(group))

    for row in group.itertuples(index=False):
        order_ids = np.fromiter(map(int, row[4].split()), dtype=np.int64)
        assert np.array_equal(np.sort(order_ids), obs["ids"]), (airline, row[1])
        key = (STAGE_RANDOM_ORDERS, AIRLINE_INDEX[airline], 9, int(row[1]))
        regenerated = obs["ids"][np.random.default_rng(np.random.SeedSequence(entropy=MASTER_SEED, spawn_key=key)).permutation(obs["n"])]
        random_rows.append({
            "Airline": airline, "Trial": int(row[1]), "Spawn key": str(key), "Seed entropy": MASTER_SEED,
            "Order SHA-256": hashlib.sha256(order_ids.astype("<i8").tobytes()).hexdigest(),
            "Matches recorded spawn key": row[3] == str(key), "Reproduced from seed": bool(np.array_equal(regenerated, order_ids))
        })

random_manifest_df = pd.DataFrame(random_rows)
random_manifest_df.to_csv(TABLES / "05_random_order_manifest.csv", index=False)

check("Every random order is reproduced from its recorded seed", random_manifest_df["Reproduced from seed"].all() and random_manifest_df["Matches recorded spawn key"].all(), len(random_manifest_df))
check("Random orders are distinct within each airline", (random_manifest_df.groupby("Airline")["Order SHA-256"].nunique() == EXPECTED_RANDOM_ORDERS).all())
check("Every randomized network uses only the airports in its airline's random orders", airport_membership_checked == len(NETWORKS) - len(AIRLINES), airport_membership_checked)
check("Random removal sets are nested across levels by construction", all(LEVEL_K[a][5] < LEVEL_K[a][10] < LEVEL_K[a][15] < LEVEL_K[a][20] for a in AIRLINES))

network_sets = network_manifest_df.groupby(["Airline", "Network set"]).size().reset_index(name="Networks")
network_sets["Paired random trials"] = EXPECTED_RANDOM_ORDERS
network_sets["Pairing"] = "Trial t uses the same saved airport order on every network of the airline"
network_sets.to_csv(TABLES / "05_random_order_pairing.csv", index=False)
display(network_sets)

,Airline,Network set,Networks,Paired random trials,Pairing
0,AA,directed,200,100,Trial t uses the same saved airport order on e...
1,AA,observed,1,100,Trial t uses the same saved airport order on e...
2,AA,reciprocal,100,100,Trial t uses the same saved airport order on e...
3,DL,directed,200,100,Trial t uses the same saved airport order on e...
4,DL,observed,1,100,Trial t uses the same saved airport order on e...
5,DL,reciprocal,100,100,Trial t uses the same saved airport order on e...
6,WN,directed,200,100,Trial t uses the same saved airport order on e...
7,WN,observed,1,100,Trial t uses the same saved airport order on e...
8,WN,reciprocal,100,100,Trial t uses the same saved airport order on e...


## Section 5.4: Observed tie-order sensitivity plan

These random orders preserve the planned sensitivity analysis on the observed networks. Each replicate permutes airports only within tied groups, and is used at all four levels as a single fixed ranking. The ID-ranking is the official primary ranking. No result is calculated here.

In [6]:
def permute_within_ties(order, values, seed_sequence):
    rng = np.random.default_rng(seed_sequence)
    permuted = order.copy()
    ordered_values = values[order]
    start = 0
    while start < len(order):
        stop = start
        while stop + 1 < len(order) and ordered_values[stop + 1] == ordered_values[start]:
            stop += 1
        if stop > start:
            permuted[start:stop + 1] = rng.permutation(order[start:stop + 1])
        start = stop + 1
    return permuted


def tie_seed(airline, strategy, replicate):
    key = (STAGE_TIE_ORDERS, AIRLINE_INDEX[airline], STRATEGY_INDEX[strategy], replicate)
    return key, np.random.SeedSequence(entropy=MASTER_SEED, spawn_key=key)


tie_sensitivity_rows = []

for airline in AIRLINES:
    obs = OBS[airline]
    ranking = RANKINGS[(airline, "observed", 0)]
    stored = {}

    for strategy in ("Degree", "Betweenness"):
        values = ranking["Degree values"] if strategy == "Degree" else ranking["Betweenness values"]
        primary = ranking[strategy]
        orders = []

        for replicate in range(TIE_ORDER_REPLICATES):
            key, seed = tie_seed(airline, strategy, replicate)
            order = permute_within_ties(primary, values, seed)
            again = permute_within_ties(primary, values, tie_seed(airline, strategy, replicate)[1])
            assert np.array_equal(order, again)
            assert len(set(order.tolist())) == obs["n"] and np.array_equal(values[order], values[primary])
            orders.append(order)
            seed_rows.append({"Stage": "tie_orders", "Airline": airline, "Strategy": strategy, "Replicate": replicate,
                              "Entropy": MASTER_SEED, "Spawn key": str(key)})

        orders = np.vstack(orders)
        stored[strategy] = orders

        for p, k in LEVEL_K[airline].items():
            sets = {tuple(sorted(row[:k].tolist())) for row in orders}
            cut = cutoff_ties(values, primary, k)
            tie_sensitivity_rows.append({
                "Airline": airline, "Strategy": strategy, "Level": p, "K": k, **cut,
                "Distinct removal sets across 100 tie orders": len(sets)
            })

    np.savez_compressed(
        PLANS / f"{airline}_observed_tie_order_sensitivity.npz",
        airport_ids=obs["ids"], replicates=np.arange(TIE_ORDER_REPLICATES),
        degree_orders=obs["ids"][stored["Degree"]], betweenness_orders=obs["ids"][stored["Betweenness"]]
    )

tie_sensitivity_df = pd.DataFrame(tie_sensitivity_rows)
tie_sensitivity_df.to_csv(TABLES / "05_tie_order_sensitivity_plan.csv", index=False)
pd.DataFrame(seed_rows).to_csv(TABLES / "05_seed_registry.csv", index=False)
check("Tie-order replicates only permute within tied groups and reproduce from their seeds", True, f"{TIE_ORDER_REPLICATES} per airline and strategy")
display(tie_sensitivity_df[tie_sensitivity_df["Tie straddles cut-off"]].round(2))

,Airline,Strategy,Level,K,Tied group at cut-off,Selected from tie,Left in tie,Tie straddles cut-off,Cut-off value is zero,Distinct removal sets across 100 tie orders
1,AA,Degree,10,23,5,2,3,True,False,10
2,AA,Degree,15,35,16,9,7,True,False,98
3,AA,Degree,20,46,14,4,10,True,False,95
5,AA,Betweenness,10,23,3,1,2,True,False,3
6,AA,Betweenness,15,35,7,4,3,True,False,33
7,AA,Betweenness,20,46,192,8,184,True,True,100
10,DL,Degree,15,32,15,1,14,True,False,15
11,DL,Degree,20,42,15,11,4,True,False,97
13,DL,Betweenness,10,21,5,1,4,True,False,5
15,DL,Betweenness,20,42,19,10,9,True,False,100


## Section 5.5: Structural check of removal application

On the first three networks of each set, each planned removal is applied and only its structure is verified: the correct number of airports remains, and no remaining route touches a removed airport. No connectivity is calculated.

In [7]:
applied = 0

for airline in AIRLINES:
    obs = OBS[airline]
    orders = [
        np.searchsorted(obs["ids"], np.fromiter(map(int, text.split()), dtype=np.int64))
        for text in orders_df.loc[orders_df["Airline"] == airline, "Airport order (BTS IDs)"]
    ]

    for model in ("observed", "directed", "reciprocal"):
        networks = sorted(key[2] for key in NETWORKS if key[0] == airline and key[1] == model)[:3]
        for network in networks:
            source, target = NETWORKS[(airline, model, network)]
            ranking = RANKINGS[(airline, model, network)]
            for p, k in LEVEL_K[airline].items():
                plans = [ranking["Degree"][:k], ranking["Betweenness"][:k], orders[0][:k]]
                for removed in plans:
                    assert len(set(removed.tolist())) == k
                    keep = np.ones(obs["n"], bool)
                    keep[removed] = False
                    assert int(keep.sum()) == obs["n"] - k
                    remaining = keep[source] & keep[target]
                    assert keep[source[remaining]].all() and keep[target[remaining]].all()
                    assert not (np.isin(source[remaining], removed).any() or np.isin(target[remaining], removed).any())
                    applied += 1

check("Removal plans apply structurally on sampled networks", applied > 0, f"{applied} applications")

checks_df = pd.DataFrame(checks)
checks_df.to_csv(TABLES / "05_setup_validation.csv", index=False)
display(checks_df)
print(f"PASS: {len(checks_df)} checks. No Step 6 resilience metric was calculated.")

PASS: 23 checks. No Step 6 resilience metric was calculated.


,Check,Result,Detail
0,AA: removal counts match the approved table,PASS,"{5: 12, 10: 23, 15: 35, 20: 46}"
1,DL: removal counts match the approved table,PASS,"{5: 11, 10: 21, 15: 32, 20: 42}"
2,WN: removal counts match the approved table,PASS,"{5: 5, 10: 10, 15: 16, 20: 21}"
3,Observed networks present,PASS,"{'AA': 1, 'DL': 1, 'WN': 1}"
4,Primary-null network counts,PASS,"{'AA': 200, 'DL': 200, 'WN': 200}"
5,Supplementary-null network counts,PASS,"{'AA': 100, 'DL': 100, 'WN': 100}"
6,Every randomized network matches its recorded ...,PASS,903
7,Every network keeps the observed in-degree and...,PASS,903
8,Step 2 observed betweenness matches a fresh ca...,PASS,
9,"Every ranking is a full airport order, descend...",PASS,1806 rankings


### Step 5 setup status

All 23 validation checks passed. The plan covers 903 networks: 3 observed, 600 primary-null and 300 supplementary-null networks. Random orders, targeted rankings and tie-order replicates were saved and reproduced identically across two complete runs.

**Tie notes for Step 6.** On the observed networks, AA at 20% betweenness selects 8 of 192 zero-betweenness airports by ascending ID. Null networks reach zero-betweenness ties earlier, especially the supplementary null: at 15% removal, the cut-off falls on a zero tie in all 100 AA supplementary networks and in 99 of 100 DL supplementary networks. The observed-network tie-order sensitivity plan is saved and has not been evaluated.

No resilience metric, observed-versus-null comparison or hypothesis verdict has been calculated. The supplementary strength-targeted strategy has not been prepared.